# 아침 모델 운영 (야간선물 → 예상 시가·노란띠·믿을 정도 → 채점)

betterkym이 고정한 모델 규칙(`workbench/morning_card.py`, `workbench/e62_band.py`)을 **고치지 않고** 그대로 매일 돌리는 노트북이다.
여기에 있는 것은 그 바깥 일뿐이다.

1. **처음 자료 받기**: 모델이 읽는 자료 폴더(`workbench/live/<이름>/data/*.csv`)를 API로 처음부터 만든다.
2. **매일 단계**: 06:20 예상 시가·노란띠(07:30 카드) → 08:11 NXT 장전 값으로 믿을 정도 등급(08:11 카드) → 09:01 실제 시가로 채점 → 20:30 장 마감 자료 이어 받기.
3. **DB 저장**: 받은 원자료와 카드·채점 결과를 서버 DB `model` 영역에 넣는다(화면은 DB만 읽는다).
4. **DB에서 자료 폴더 다시 만들기**: 서버 폴더가 지워져도 DB로 되살린다.

서버(오라클)에서 `run_model.py`가 이 노트북의 함수를 불러 일정대로 돌린다. 처리 코드는 모두 이 노트북에 있다.

**종목 3구분** (`data/service_universe.csv`)
- 1구분(tier 1): 모델 설계에 쓰인 200종목(코스피 150 + 코스닥 50). 별도 표시 없음.
- 2구분(tier 2): 서비스 대상(코스피 전체 + 코스닥150) 중 나머지 755종목. 같은 규칙으로 계산하고 "모델 설계에 쓰이지 않은 종목" 경고를 단다.
- 3구분: 그 밖의 종목. 계산하지 않고 화면에서 "서비스 대상이 아니에요"로 보인다.

**설계 종목을 200종목으로 고정하는 이유**: 노란띠 폭 규칙과 믿을 정도 계수를 학습하는 "설계 종목"은 `e62_band.split_stocks`가
종목 목록을 고정 씨앗으로 섞어 반으로 나눠 정한다. 목록에 955종목을 넣으면 나누는 결과가 달라져 1구분 값도 바뀐다.
그래서 나누기는 언제나 원래 200종목 목록으로 하고, 그렇게 학습한 규칙을 955종목 모두에 적용한다(1구분 값은 betterkym 결과와 같다).

## 1. 설정
- `MODEL_ENV_FILE`: 열쇠 파일. 서버는 `~/overnight/.env`(KIS_APP_KEY, KIS_APP_SECRET, DART_API_KEY, GNEWS_APP_DB_URL, ALERT_TELEGRAM_*). 값에 따옴표를 쓰지 않는다(모델 코드가 큰따옴표만 벗긴다).
- 한국투자증권(KIS) 조회는 초당 `KIS_RPS`번을 넘지 않게 여러 갈래로 나눠 보낸다(계좌 한도 초당 20번).

In [ ]:
import fcntl, hashlib, importlib.util, json, os, shutil, sys, threading, time, traceback, urllib.parse, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

MODEL = Path(os.environ.get('MODEL_DIR', Path.cwd())).resolve()      # 이 노트북이 있는 폴더(overnight_briefing/model)
sys.path[:0] = [p for p in (str(MODEL), str(MODEL / 'src')) if p not in sys.path]
LIVE = MODEL / 'workbench/live'                                       # 날짜별 자료 폴더(live_refresh와 같은 자리)
CARDS = MODEL / 'workbench/runs/claude_live_r1'                       # 카드·채점 파일(morning_card 기본 자리)
ENV_FILE = Path(os.environ.get('MODEL_ENV_FILE', MODEL.parents[1] / '.env')).expanduser()
TIER1_FILE = MODEL / 'data/universe.csv'                              # 모델 설계 200종목(betterkym)
SERVICE_FILE = MODEL / 'data/service_universe.csv'                    # 1·2구분 955종목
DB_ENV = 'GNEWS_APP_DB_URL'                                           # 서버 DB(model 영역에 쓰기)
BOOT_NAME = '00000000boot'      # 처음 자료 폴더 이름. 날짜 폴더보다 이름이 앞이라 live_refresh가 이어받을 폴더로 고른다
HISTORY_START = '20241201'      # 일봉 시작(선물 근월물 표의 첫 계약 20241213 직전부터)
NXT_START = '20250304'          # NXT 개장일. 설계 종목(믿을 정도 학습에 쓰는 107종목)은 이날부터
NXT_RECENT_DAYS = 45            # 나머지 종목은 최근 45거래일(종목별 08:11 오차 크기 계산에 최소 20일 필요)
NIGHT_MINUTE_DAYS = 400         # 야간선물 1분봉: KIS가 약 1년 치만 준다. 그 안에서 받을 수 있는 만큼
DART_START = '20261001'         # 공시 목록 시작(실적 공시 다음 날 주의 문구에는 최근 것만 필요)
KIS_RPS = 15
KIS_WORKERS = 12              # 서버→KIS 응답이 느려(약 0.6초) 여러 갈래로 보낸다. 초당 한도는 KIS_RPS가 지킨다
KEEP_FOLDERS = 14               # 날짜 자료 폴더는 최근 14개만 남긴다(전부 DB에 있다)
KST = 'Asia/Seoul'


def read_env(path=ENV_FILE):
    env = {}
    if Path(path).exists():
        for line in Path(path).read_text(encoding='utf-8').splitlines():
            if '=' in line and not line.lstrip().startswith('#'):
                k, v = line.split('=', 1)
                env[k.strip()] = v.strip().strip('"').strip("'")
    return env


ENV = read_env()
now_kst = lambda: pd.Timestamp.now(tz=KST)
today = lambda: now_kst().strftime('%Y%m%d')
print('모델 폴더', MODEL, '/ 열쇠 파일', ENV_FILE, '있음' if ENV_FILE.exists() else '없음')

## 2. 종목 구분과 KIS 조회 도구
- `model_universe()`: 모델 자료 폴더의 `universe.csv`(Code·Name·Marcap·Market·Industry·group·etf 칸)를 955종목으로 만든다. 1구분은 betterkym 표 그대로, 2구분은 업종(group)만 서비스 표에서 채운다(모델 계산에는 쓰이지 않는 칸).
- `kis_call()`: betterkym의 `live_refresh.kis_get`(실패 시 다시 시도, 한도 초과 시 1초 쉼)을 그대로 쓰고, 여러 갈래에서 불러도 초당 `KIS_RPS`번을 넘지 않게 줄 세운다.
- `fast_premarket()`, `fast_stocks()`, `fast_opens()`: betterkym의 `refresh_premarket`, `refresh_stocks`, `check_open.opens`와 **같은 조회·같은 결과 모양**이고, 여러 갈래로 보내 빠를 뿐이다. 955종목을 08:11 직후 1분 안에 받으려고 바꿔 끼운다.

In [ ]:
from collect import KIS
from workbench import e62_band as e62
from workbench import live_panel as lp
from workbench import live_refresh as lr

TIER1 = sorted(pd.read_csv(TIER1_FILE, dtype={'Code': str}).Code)
SERVICE = pd.read_csv(SERVICE_FILE, dtype={'code': str})


def model_universe():
    t1 = pd.read_csv(TIER1_FILE, dtype={'Code': str})
    t2 = SERVICE[SERVICE.tier == 2]
    add = pd.DataFrame(dict(Code=t2.code, Name=t2.name, Marcap=np.nan, Market=t2.market, Industry='', group=t2.sector, etf=''))
    return pd.concat([t1, add], ignore_index=True)


class Rate:
    """여러 갈래가 함께 써도 초당 rps번을 넘지 않게 차례를 나눠 준다."""
    def __init__(self, rps):
        self.gap, self.lock, self.next = 1 / rps, threading.Lock(), 0.0

    def wait(self):
        with self.lock:
            now = time.monotonic()
            t = max(now, self.next)
            self.next = t + self.gap
        time.sleep(max(0.0, t - now))


RATE = Rate(KIS_RPS)


def kis_call(kis, path, tr, params):
    RATE.wait()
    return lr.kis_get(kis, path, tr, params)


def pmap(fn, items, workers=KIS_WORKERS, label=None):
    out, n = [], len(items)
    with ThreadPoolExecutor(workers) as ex:
        for i, r in enumerate(ex.map(fn, items), 1):
            out.append(r)
            if label and (i % 2000 == 0 or i == n):
                print(f'  {label} {i}/{n}', flush=True)
    return out


def premarket_rows(kis, code, day):
    res = kis_call(kis, '/uapi/domestic-stock/v1/quotations/inquire-time-dailychartprice', 'FHKST03010230',
                   {'FID_COND_MRKT_DIV_CODE': 'NX', 'FID_INPUT_ISCD': code, 'FID_INPUT_HOUR_1': '081000', 'FID_INPUT_DATE_1': day,
                    'FID_PW_DATA_INCU_YN': 'N', 'FID_FAKE_TICK_INCU_YN': ''})
    return [dict(day=day, code=code, hhmmss=r['stck_cntg_hour'], price=float(r['stck_prpr']), volume=float(r['cntg_vol']))
            for r in res.get('output2') or [] if r.get('stck_bsop_date') == day and '080000' <= r['stck_cntg_hour'] <= '081000']


def fast_premarket(kis, df, codes, days):
    """live_refresh.refresh_premarket과 같은 결과(df, 새 줄 수)."""
    rows = sum(pmap(lambda p: premarket_rows(kis, *p), [(c, d) for d in days for c in codes], label='NXT 장전'), [])
    new = pd.DataFrame(rows, columns=['day', 'code', 'hhmmss', 'price', 'volume'])
    keys = ['day', 'code', 'hhmmss']
    new = new.drop_duplicates(keys)
    if new.empty:
        return df, 0
    return pd.concat([df, new[~new.set_index(keys).index.isin(df.set_index(keys).index)]], ignore_index=True), len(new)


def stock_rows(kis, code, start, end):
    res = kis_call(kis, '/uapi/domestic-stock/v1/quotations/inquire-daily-itemchartprice', 'FHKST03010100',
                   {'FID_COND_MRKT_DIV_CODE': 'J', 'FID_INPUT_ISCD': code, 'FID_INPUT_DATE_1': start, 'FID_INPUT_DATE_2': end,
                    'FID_PERIOD_DIV_CODE': 'D', 'FID_ORG_ADJ_PRC': '0'})
    return [dict(date=r['stck_bsop_date'], Open=float(r['stck_oprc']), Close=float(r['stck_clpr']), code=code)
            for r in res.get('output2') or [] if r.get('stck_bsop_date') and start <= r['stck_bsop_date'] <= end and float(r['stck_oprc']) > 0]


def fast_stocks(kis, df, codes, start, last_close_day):
    """live_refresh.refresh_stocks와 같은 결과. 한 번에 100줄까지라 기간이 길면 130일씩 나눠 받는다."""
    spans, a = [], pd.Timestamp(start)
    while a <= pd.Timestamp(last_close_day):
        b = min(a + pd.Timedelta(days=130), pd.Timestamp(last_close_day))
        spans.append((a.strftime('%Y%m%d'), b.strftime('%Y%m%d')))
        a = b + pd.Timedelta(days=1)
    rows = sum(pmap(lambda p: stock_rows(kis, p[0], *p[1]), [(c, s) for s in spans for c in codes], label='종목 일봉'), [])
    new = pd.DataFrame(rows, columns=['date', 'Open', 'Close', 'code']).drop_duplicates(['date', 'code'])
    if new.empty:
        return df
    old = df.set_index(['date', 'code']).index
    return pd.concat([df, new[~new.set_index(['date', 'code']).index.isin(old)]], ignore_index=True).sort_values(['code', 'date'])


def fast_opens(kis, codes, day):
    """check_open.opens와 같은 결과({종목: 그날 시가})."""
    def one(code):
        res = kis_call(kis, '/uapi/domestic-stock/v1/quotations/inquire-daily-itemchartprice', 'FHKST03010100',
                       {'FID_COND_MRKT_DIV_CODE': 'J', 'FID_INPUT_ISCD': code, 'FID_INPUT_DATE_1': day, 'FID_INPUT_DATE_2': day,
                        'FID_PERIOD_DIV_CODE': 'D', 'FID_ORG_ADJ_PRC': '0'})
        for r in res.get('output2') or []:
            if r.get('stck_bsop_date') == day and float(r.get('stck_oprc') or 0) > 0:
                return code, float(r['stck_oprc'])
        return code, None
    return {c: o for c, o in pmap(one, list(codes)) if o is not None}


_split = e62.split_stocks


def patch_model():
    """빠른 조회로 바꿔 끼우고, 설계 종목 나누기를 200종목 목록으로 고정한다(계산 규칙은 그대로)."""
    lr.refresh_premarket = fast_premarket
    lr.refresh_stocks = lambda kis, df, codes, start, last: fast_stocks(kis, df, codes, start, last)
    e62.split_stocks = lambda codes, dev: _split(TIER1, dev)

## 3. 처음 자료 받기 (한 번)
`workbench/live/00000000boot/`에 모델이 읽는 파일을 모두 만든다. 다음 날부터 `live_refresh`가 이 폴더를 이어받는다.

| 파일 | 받는 곳 | 기간 |
|---|---|---|
| futures_daily.csv | KIS 선물 일봉(주간 F, 야간 CM), 근월물 이어 붙이기(`collect.collect_futures`) | 2024-12-13 ~ |
| night_minutes.csv | KIS 야간선물 1분봉(`live_refresh.night_minutes`) | KIS가 주는 최근 약 1년 |
| universe_daily.csv | KIS 종목 일봉 시가·종가, 955종목 | 2024-12-01 ~ |
| universe_premarket.csv | KIS NXT 08:00~08:10 1분봉 | 설계 종목 107개: 2025-03-04 ~ / 나머지: 최근 45거래일 |
| kr_daily.csv, ks200_long.csv | 코스피200 지수(FinanceDataReader) | 2024-11 ~ / 2013-06 ~ |
| us_hourly.csv, vix_daily.csv | yfinance 미국 선물·환율 시간봉, VIX | 최근 730일 |
| investor_daily.csv | (투자자별 매매. 지금 규칙(S1·D08)에 쓰이지 않아 값이 빈 자리 표시 한 줄) | - |
| nxt_premarket.csv | (betterkym 개발용 14종목 원본. 같은 자료를 universe_premarket에 넣어 빈 표) | - |
| fred_release_dates.csv, fomc_dates.csv | 저장소 `data/` 그대로 | - |
| ../dart_filings.csv | DART 공시 목록(`live_refresh.refresh_dart`) | 2026-10-01 ~ |

모델 규칙이 실제로 쓰는 입력: 노란띠(S1)는 야간선물 일봉으로 만든 |야간 수익률|과 종목 일봉, 믿을 정도(D08)는 NXT 08:10 값·예상 시가 크기·평소 오차·회귀 설명력. 1분봉·미국 시간봉·VIX는 패널의 다른 칸(지금 규칙에서 안 씀)과 화면의 밤사이 흐름에 쓰인다.

In [ ]:
def last_finished_day(now=None):
    """주간장이 끝난 마지막 날짜(16:00 전이면 어제까지)."""
    now = now or now_kst()
    d = now if now.hour >= 16 else now - pd.Timedelta(days=1)
    return d.strftime('%Y%m%d')


def save_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f'  {path.name}: {len(df):,}줄', flush=True)


def investor_placeholder():
    """투자자별 매매(investor_daily.csv) 자리. 지금 규칙(S1·D08)에 쓰이지 않아 값은 비운다.
    줄이 하나도 없으면 숫자 칸이 글자 칸으로 읽혀 패널 계산이 멈추므로, 값이 빈 숫자 한 줄을 둔다(어느 종목·날짜와도 맞지 않음)."""
    return pd.DataFrame([dict(date='19000101', code='000000', value=np.nan, frgn=np.nan, orgn=np.nan)])


def boot_ks200(start='2013-06-01'):
    import FinanceDataReader as fdr
    k = fdr.DataReader('KS200', start).reset_index()
    k['date'] = pd.to_datetime(k.iloc[:, 0]).dt.strftime('%Y%m%d')
    kr = pd.DataFrame(dict(date=k.date, Open=k.Open, High=k.High, Low=k.Low, Close=k.Close, Volume=k.Volume, code='KS200', name='코스피200 지수'))
    long = k[['date', 'Open', 'Close']].rename(columns={'date': 'Date'})
    return kr[kr.date >= '20241101'].reset_index(drop=True), long


def boot_vix():
    import yfinance as yf
    d = yf.download('^VIX', period='730d', interval='1d', progress=False, auto_adjust=False)
    d.columns = [c[0] if isinstance(c, tuple) else c for c in d.columns]
    d.index = pd.to_datetime(d.index).strftime('%Y%m%d')
    return d[['Close']].rename_axis('us_date').reset_index()


def boot_night_minutes(kis, fut, until):
    nights = fut[(fut.session == 'night') & (fut.date <= until)].sort_values('date')
    nights = nights[nights.date >= (pd.Timestamp(until) - pd.Timedelta(days=NIGHT_MINUTE_DAYS)).strftime('%Y%m%d')]
    def one(r):
        RATE.wait()
        return lr.night_minutes(kis, r.date, r.code)
    frames = [f for f in pmap(one, list(nights.itertuples()), workers=3, label='야간 1분봉') if len(f)]
    nm = pd.concat(frames, ignore_index=True).sort_values(['date', 'hhmmss'], kind='stable') if frames else \
        pd.DataFrame(columns=['date', 'code', 'hhmmss', 'price', 'volume'])
    print(f'  야간 1분봉: {nm.date.nunique()}밤 받음({nm.date.min()} ~ {nm.date.max()}), 요청한 밤 {len(nights)}', flush=True)
    return nm


def bootstrap(until=None, name=BOOT_NAME, steps=None):
    """처음 자료 폴더 만들기. steps로 일부만 다시 받을 수 있다(예: steps={'premarket'})."""
    until = until or last_finished_day()
    out, data = LIVE / name, LIVE / name / 'data'
    data.mkdir(parents=True, exist_ok=True)
    want = lambda s: steps is None or s in steps
    kis = KIS(str(ENV_FILE))
    started = now_kst()
    uni = model_universe()
    codes = uni.Code.tolist()
    save_csv(uni, data / 'universe.csv')
    for n in ('fred_release_dates.csv', 'fomc_dates.csv'):
        shutil.copy2(MODEL / 'data' / n, data / n)
    if want('futures'):
        import collect
        fut = collect.collect_futures(str(ENV_FILE), until)
        fut = fut[[lr.finished(s, d, started) for s, d in zip(fut.session, fut.date)]]
        save_csv(fut, data / 'futures_daily.csv')
    fut = pd.read_csv(data / 'futures_daily.csv', dtype=lp.TYPES)
    if want('night'):
        save_csv(boot_night_minutes(kis, fut, until), data / 'night_minutes.csv')
    if want('stocks'):
        empty = pd.DataFrame(columns=['date', 'Open', 'Close', 'code'])
        save_csv(fast_stocks(kis, empty, codes, HISTORY_START, until), data / 'universe_daily.csv')
    if want('premarket'):
        days = sorted(fut[(fut.session == 'day') & (fut.date >= NXT_START) & (fut.date <= until)].date.unique())
        empty = pd.DataFrame(columns=['day', 'code', 'hhmmss', 'price', 'volume'])
        dev = json.loads((MODEL / 'workbench/protocol_i24_features.json').read_text())['development_codes']
        design, _ = _split(TIER1, dev)      # 믿을 정도(D08)는 설계 종목의 과거 행으로만 학습한다
        rest = [c for c in codes if c not in set(design)]
        pm, _ = fast_premarket(kis, empty, design, days)
        pm, _ = fast_premarket(kis, pm, rest, days[-NXT_RECENT_DAYS:])
        save_csv(pm.sort_values(['day', 'code', 'hhmmss']), data / 'universe_premarket.csv')
        save_csv(pd.DataFrame(columns=['date', 'day', 'code', 'hhmmss', 'price', 'volume']), data / 'nxt_premarket.csv')
    if want('market'):
        kr, long = boot_ks200()
        save_csv(kr, data / 'kr_daily.csv')
        save_csv(long, data / 'ks200_long.csv')
        import collect
        save_csv(collect.collect_us_hourly(), data / 'us_hourly.csv')
        save_csv(boot_vix(), data / 'vix_daily.csv')
        save_csv(investor_placeholder(), data / 'investor_daily.csv')
    if want('dart'):
        cols = ['code', 'corp_code', 'rcept_no', 'rcept_dt', 'report_nm', 'corp_cls', 'flr_nm', 'rm']
        empty = out / 'dart_empty.csv'
        pd.DataFrame(columns=cols).to_csv(empty, index=False)
        filings, n = lr.refresh_dart(str(ENV_FILE), set(codes), DART_START, until, old_path=empty)
        empty.unlink()
        save_csv(filings, out / 'dart_filings.csv')
    log = dict(started_at=started.isoformat(timespec='seconds'), kind='bootstrap', base='API에서 처음 받음',
               trading_day=dict(date=until, open=True, checked_by='bootstrap'),
               futures_last={s: fut[fut.session == s].date.max() for s in ('day', 'night')},
               inputs_sha256={n: lp.sha(data / n) for n in lp.INPUTS if (data / n).exists()},
               finished_at=now_kst().isoformat(timespec='seconds'))
    (out / 'refresh_manifest.json').write_text(json.dumps(log, ensure_ascii=False, indent=1))
    missing = [n for n in lp.INPUTS if not (data / n).exists()]
    print('처음 자료 끝', until, '빠진 파일:', missing or '없음', flush=True)
    return out

## 4. DB 저장
서버 DB의 `model` 영역에 표를 만들고, 자료 폴더의 파일과 카드·채점 결과를 넣는다. 같은 열쇠(날짜·종목·시각)는 새 값으로 덮어써서 몇 번을 넣어도 같다.

| 표 | 내용 |
|---|---|
| model.stocks | 955종목과 구분(tier 1·2) |
| model.futures_daily, model.night_minutes | 야간선물 일봉·1분봉 |
| model.stock_daily, model.premarket, model.ks200 | 종목 일봉 시가·종가, NXT 장전 1분봉, 코스피200 |
| model.filings | DART 공시 목록 |
| model.cards, model.card_stocks | 아침 카드(07:30·08:11)와 종목별 예상 시가 H·노란띠 80%/90%·믿을 정도 |
| model.checks, model.check_stocks | 개장 뒤 채점(띠 안에 들었는지 등) |
| model.runs | 단계별 실행 기록(성공·실패) |

In [ ]:
import psycopg

# model 영역은 서버 관리자 계정이 한 번 만들어 둔다(gnews_app 소유, gnews_read 읽기):
#   create schema model authorization gnews_app; grant usage on schema model to gnews_read;
#   alter default privileges for role gnews_app in schema model grant select on tables to gnews_read;
DDL = """
create table if not exists model.stocks (code text primary key, name text, market text, sector text, tier int);
create table if not exists model.futures_daily (date text, session text, code text, open float8, high float8, low float8, close float8, volume float8, primary key (date, session));
create table if not exists model.night_minutes (date text, code text, hhmmss text, price float8, volume float8, primary key (date, code, hhmmss));
create table if not exists model.stock_daily (code text, date text, open float8, close float8, primary key (code, date));
create table if not exists model.premarket (day text, code text, hhmmss text, price float8, volume float8, primary key (day, code, hhmmss));
create table if not exists model.ks200 (date text primary key, open float8, high float8, low float8, close float8, volume float8);
create table if not exists model.filings (rcept_no text primary key, code text, corp_code text, rcept_dt text, report_nm text, corp_cls text, flr_nm text, rm text);
create table if not exists model.cards (t text, slot text, status text, reason text, reason_text text, d text, x float8, issued_at timestamptz,
  timing_status text, readiness jsonb, grade_status jsonb, cautions jsonb, n_stocks int, n_graded int, file text, sha256 text, card jsonb, primary key (t, slot));
create table if not exists model.card_stocks (t text, slot text, code text, name text, pc float8, h float8, l80 float8, u80 float8, l90 float8, u90 float8,
  grade_level text, grade_p float8, nxt_g float8, cautions jsonb, primary key (t, slot, code));
create table if not exists model.checks (t text, slot text, scored int, in_band80 float8, in_band90 float8, mae_h float8, dir_nxt_all float8, summary jsonb, primary key (t, slot));
create table if not exists model.check_stocks (t text, slot text, code text, open float8, y float8, in80 bool, in90 bool, above bool, below bool, primary key (t, slot, code));
create table if not exists model.runs (id bigserial primary key, step text, t text, started_at timestamptz, finished_at timestamptz, ok bool, message text);
create index if not exists card_stocks_code on model.card_stocks (code, t);
create table if not exists model.us_daily (symbol text, us_date text, name text, open float8, high float8, low float8, close float8, prev_close float8,
  updated_at timestamptz default now(), primary key (symbol, us_date));
create table if not exists model.us_intraday (symbol text, us_date text, ts timestamptz, price float8, primary key (symbol, ts));
"""


def db_url():
    url = os.environ.get(DB_ENV) or ENV.get(DB_ENV)
    if not url:
        raise SystemExit(f'{DB_ENV}가 열쇠 파일에 없다')
    return url.replace('postgresql+psycopg://', 'postgresql://')


def db():
    return psycopg.connect(db_url(), autocommit=False)


def ensure_tables():
    with db() as con:
        con.execute(DDL)


def upsert(table, df, keys, chunk=50000):
    """df를 임시 표로 복사한 뒤 table에 넣는다(같은 열쇠는 새 값으로)."""
    if df is None or df.empty:
        return 0
    cols = list(df.columns)
    rest = [c for c in cols if c not in keys]
    clean = df.astype(object).where(pd.notna(df), None)
    with db() as con, con.cursor() as cur:
        cur.execute(f'create temp table _up (like {table} including defaults) on commit drop')
        for i in range(0, len(clean), chunk):
            with cur.copy(f'copy _up ({", ".join(cols)}) from stdin') as cp:
                for row in clean.iloc[i:i + chunk].itertuples(index=False):
                    cp.write_row(row)
        upd = ', '.join(f'{c} = excluded.{c}' for c in rest) if rest else None
        cur.execute(f'insert into {table} ({", ".join(cols)}) select {", ".join(cols)} from _up '
                    f'on conflict ({", ".join(keys)}) do ' + (f'update set {upd}' if upd else 'nothing'))
        return len(clean)


def save_folder_db(folder, since=None):
    """자료 폴더의 원자료를 DB에 넣는다. since(YYYYMMDD)가 있으면 그날부터만."""
    ensure_tables()
    folder = Path(folder)
    data = folder / 'data'
    read = lambda n: pd.read_csv(data / n, dtype=lp.TYPES) if (data / n).exists() else None
    cut = lambda df, col: df if since is None or df is None else df[df[col] >= since]
    done = {}
    done['stocks'] = upsert('model.stocks', SERVICE[['code', 'name', 'market', 'sector', 'tier']], ['code'])
    f = read('futures_daily.csv')
    done['futures_daily'] = upsert('model.futures_daily', cut(f, 'date'), ['date', 'session'])
    nm = read('night_minutes.csv')
    done['night_minutes'] = upsert('model.night_minutes', cut(nm, 'date'), ['date', 'code', 'hhmmss'])
    ud = read('universe_daily.csv')
    if ud is not None:
        done['stock_daily'] = upsert('model.stock_daily', cut(ud, 'date').rename(columns={'Open': 'open', 'Close': 'close'})[['code', 'date', 'open', 'close']], ['code', 'date'])
    pm = read('universe_premarket.csv')
    done['premarket'] = upsert('model.premarket', cut(pm, 'day'), ['day', 'code', 'hhmmss'])
    kr = read('kr_daily.csv')
    if kr is not None:
        k = cut(kr[kr.code == 'KS200'], 'date')
        done['ks200'] = upsert('model.ks200', k[['date', 'Open', 'High', 'Low', 'Close', 'Volume']].set_axis(['date', 'open', 'high', 'low', 'close', 'volume'], axis=1), ['date'])
    if (folder / 'dart_filings.csv').exists():
        fl = pd.read_csv(folder / 'dart_filings.csv', dtype=str)
        done['filings'] = upsert('model.filings', cut(fl, 'rcept_dt'), ['rcept_no'])
    print('DB 저장', folder.name, {k: v for k, v in done.items() if v}, flush=True)
    return done


def save_card_db(path):
    path = Path(path)
    card = json.loads(path.read_text())
    sha = Path(str(path) + '.sha256').read_text().split()[0] if Path(str(path) + '.sha256').exists() else None
    t, slot = card['T'], card.get('slot')
    stocks = card.get('stocks', [])
    row = pd.DataFrame([dict(t=t, slot=slot, status=card.get('status'), reason=card.get('reason'), reason_text=card.get('reason_text'), d=card.get('D'),
                             x=(card.get('market') or {}).get('x'), issued_at=card.get('issued_at'), timing_status=card.get('timing_status'),
                             readiness=json.dumps(card.get('data_readiness'), ensure_ascii=False), grade_status=json.dumps(card.get('grade_status'), ensure_ascii=False),
                             cautions=json.dumps((card.get('market') or {}).get('cautions', []), ensure_ascii=False), n_stocks=len(stocks),
                             n_graded=sum(s.get('grade_0811') is not None for s in stocks), file=path.name, sha256=sha,
                             card=json.dumps({k: v for k, v in card.items() if k not in ('stocks', 'inputs')}, ensure_ascii=False))])
    ensure_tables()
    upsert('model.cards', row, ['t', 'slot'])
    rows = [dict(t=t, slot=slot, code=s['code'], name=s['name'], pc=s['base_close']['value'], h=s['H'], l80=s['band80'][0], u80=s['band80'][1],
                 l90=s['band90'][0], u90=s['band90'][1], grade_level=(s.get('grade_0811') or {}).get('level'), grade_p=(s.get('grade_0811') or {}).get('p'),
                 nxt_g=(s.get('grade_0811') or {}).get('nxt_g'), cautions=json.dumps(s.get('cautions', []), ensure_ascii=False)) for s in stocks]
    with db() as con:
        con.execute('delete from model.card_stocks where t = %s and slot = %s', (t, slot))
    upsert('model.card_stocks', pd.DataFrame(rows), ['t', 'slot', 'code'])
    print('카드 DB 저장', t, slot, card.get('status'), '종목', len(rows), flush=True)


def save_check_db(path):
    c = json.loads(Path(path).read_text())
    slot = '0811' if '_0811' in c['card'] else '0730'
    ensure_tables()
    upsert('model.checks', pd.DataFrame([dict(t=c['date'], slot=slot, scored=c['stocks_scored'], in_band80=c['in_band80'], in_band90=c['in_band90'],
                                              mae_h=c['mae_H'], dir_nxt_all=c['dir_nxt_all'],
                                              summary=json.dumps({k: v for k, v in c.items() if k != 'rows'}, ensure_ascii=False, default=float))]), ['t', 'slot'])
    rows = pd.DataFrame(c['rows'])
    if len(rows):
        rows = rows.assign(t=c['date'], slot=slot)[['t', 'slot', 'code', 'open', 'y', 'in80', 'in90', 'above', 'below']]
        upsert('model.check_stocks', rows, ['t', 'slot', 'code'])
    print('채점 DB 저장', c['date'], slot, '띠 안', c['in_band80'], '%', flush=True)

## 5. 매일 단계
betterkym의 운영 순서(`seal_morning.sh`, `after_close.sh`)와 같다. 시각은 서버 cron이 맞춘다(한국 시각).

| 단계 | 시각 | 하는 일 |
|---|---|---|
| `morning0730` | 06:20 | 끝난 야간장·공시 이어 받기 → 07:30 카드(예상 시가 H, 노란띠) → 지난밤 1분봉 → DB |
| `morning0811` | 08:11:30 | NXT 08:00~08:10 955종목 받기 → 08:11 카드(믿을 정도 등급으로 보정) → DB |
| `open` | 09:01, 10:31 | 그날 시가로 두 카드 채점(카드 해시가 같은지 먼저 확인) → DB. 수능일처럼 10시 개장이면 10:31에 채점 |
| `close` | 20:30 | 그날 주간장·종목 일봉·코스피200 이어 받기(다음 날 아침의 바탕) → DB → 오래된 폴더 정리. 16:00~20:00 애프터마켓 중에는 KIS 종가 칸이 시간외 가격이라 끝난 뒤에 받는다(betterkym 16:05 기록은 11-12 카드에서 168종목 전일 종가가 시간외 가격이었음) |

휴장일이면 `live_refresh`가 휴장으로 기록하고 카드는 "KRX 휴장일" 상태로 남는다.

In [ ]:
def run_main(fn, argv):
    old = sys.argv
    sys.argv = ['x'] + [str(a) for a in argv]
    try:
        fn()
    finally:
        sys.argv = old


def newest_card(T, slot):
    files = sorted(CARDS.glob(f'card_{T}_{slot}*.json'), key=lambda p: p.stat().st_mtime)
    return files[-1] if files else None


def make_card(T, slot):
    from workbench import morning_card
    run_main(morning_card.main, ['--date', T, '--slot', slot, '--root', LIVE / T, '--events', LIVE / T / 'dart_filings.csv'])
    path = newest_card(T, slot)
    save_card_db(path)
    return path


def step_morning(T, slot):
    patch_model()
    argv = ['--name', T, '--env', ENV_FILE, '--until', T, '--no-night']
    if slot == '0811':
        argv += ['--premarket-days', T]
    run_main(lr.main, argv)
    manifest = json.loads((LIVE / T / 'refresh_manifest.json').read_text())
    path = make_card(T, slot)
    if manifest.get('skipped'):
        return f'휴장일: {manifest["skipped"]}'
    if slot == '0730':
        run_main(lr.main, ['--name', T, '--env', ENV_FILE, '--until', T, '--night-only'])
    save_folder_db(LIVE / T, since=(pd.Timestamp(T) - pd.Timedelta(days=10)).strftime('%Y%m%d'))
    card = json.loads(path.read_text())
    return f'{slot} 카드 {card["status"]} 종목 {len(card["stocks"])} 등급 {sum(s.get("grade_0811") is not None for s in card["stocks"])}'


def load_check_open():
    spec = importlib.util.spec_from_file_location('check_open', CARDS / 'check_open.py')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    mod.opens = fast_opens
    return mod


def step_open(T):
    mod, done = load_check_open(), []
    for slot in ('0730', '0811'):
        card, out = newest_card(T, slot), CARDS / f'check_{T}_{slot}.json'
        if card is None or out.exists():
            continue
        if json.loads(card.read_text()).get('status') != 'ok':
            continue
        try:
            run_main(mod.main, ['--date', T, '--env', ENV_FILE, '--card', card, '--out', out])
        except (KeyError, ValueError) as err:          # 아직 시가가 없으면(10시 개장 등) 채점 줄이 비어 멈춘다 → 다음 시각에 다시
            done.append(f'{slot} 시가 아직 없음({type(err).__name__})')
            continue
        save_check_db(out)
        done.append(f'{slot} 채점')
    return ', '.join(done) or '채점할 카드 없음'


def cleanup_folders():
    names = sorted(p for p in LIVE.iterdir() if p.is_dir() and (p / 'refresh_manifest.json').exists())
    for p in names[:-KEEP_FOLDERS]:
        shutil.rmtree(p)
    return len(names[:-KEEP_FOLDERS])


def step_close(T):
    patch_model()
    run_main(lr.main, ['--name', T + 'c', '--env', ENV_FILE, '--until', T])
    folder = LIVE / (T + 'c')
    if json.loads((folder / 'refresh_manifest.json').read_text()).get('skipped'):
        shutil.rmtree(folder)
        return '휴장일'
    save_folder_db(folder, since=(pd.Timestamp(T) - pd.Timedelta(days=10)).strftime('%Y%m%d'))
    return f'장 마감 자료 저장, 지운 옛 폴더 {cleanup_folders()}개'

## 5-1. 밤사이 화면용 실시간 자료
- `night_live`(한국 18:00~06:10, 10분마다): 지금 진행 중인 야간선물 1분봉과 그날 주간 종가를 DB에 넣는다. 07:30 전·주말 화면의 "진행 중" 흐름용이다. 모델 계산은 06:20 단계가 자료 폴더로 따로 한다.
- `us_market`(미국 정규장 중 30분마다, 끝난 뒤 한 번): 다우·S&P 500·나스닥·원/달러 환율의 5분봉과 시가·고가·저가·종가·전일 종가(yfinance). 홈의 미국 시장 칸과 미국시장 지표 상세용이다.

In [ ]:
US_SYMBOLS = {'^DJI': '다우존스', '^GSPC': 'S&P 500', '^IXIC': '나스닥', 'KRW=X': '원/달러 환율'}


def step_night_live(T=None):
    now = now_kst()
    if not (now.hour >= 18 or now.hour < 6 or (now.hour == 6 and now.minute <= 10)):
        return '야간장 시간이 아님'
    D = (now if now.hour >= 18 else now - pd.Timedelta(days=1)).strftime('%Y%m%d')
    if pd.Timestamp(D).weekday() >= 5:
        return '주말 밤(야간장 없음)'
    code = lr.front(D)
    kis = KIS(str(ENV_FILE))
    day = [r for r in kis.daily('F', code, D, D) if r.get('stck_bsop_date') == D]
    if not day:
        return f'{D} 주간장 없음(휴장)'
    r = day[0]
    ensure_tables()
    upsert('model.futures_daily', pd.DataFrame([dict(date=D, session='day', code=code, open=float(r['futs_oprc']), high=float(r['futs_hgpr']),
                                                     low=float(r['futs_lwpr']), close=float(r['futs_prpr']), volume=float(r.get('acml_vol') or 0))]), ['date', 'session'])
    nm = lr.night_minutes(kis, D, code)
    n = upsert('model.night_minutes', nm, ['date', 'code', 'hhmmss'])
    return f'{D} 야간 1분봉 {n}개(마지막 {nm.hhmmss.max() if len(nm) else "-"})'


def step_us_market(T=None):
    import yfinance as yf
    ensure_tables()
    out = []
    for sym, name in US_SYMBOLS.items():
        m = yf.download(sym, period='5d', interval='5m', progress=False, auto_adjust=False)
        d = yf.download(sym, period='15d', interval='1d', progress=False, auto_adjust=False)
        for f in (m, d):
            f.columns = [c[0] if isinstance(c, tuple) else c for c in f.columns]
        if m.empty or d.empty:
            out.append(f'{name} 없음')
            continue
        idx = pd.to_datetime(m.index)
        m.index = (idx.tz_localize('UTC') if idx.tz is None else idx).tz_convert('America/New_York')
        last = m.index[-1].date()
        today = m[m.index.date == last]
        upsert('model.us_intraday', pd.DataFrame(dict(symbol=sym, us_date=last.strftime('%Y%m%d'), ts=[t.tz_convert('UTC') for t in today.index],
                                                     price=today.Close.astype(float).to_numpy())), ['symbol', 'ts'])
        d.index = pd.to_datetime(d.index).strftime('%Y%m%d')
        dd = d[['Open', 'High', 'Low', 'Close']].astype(float).assign(prev=lambda x: x.Close.shift(1)).dropna(subset=['prev'])
        upsert('model.us_daily', pd.DataFrame(dict(symbol=sym, us_date=dd.index, name=name, open=dd.Open, high=dd.High, low=dd.Low, close=dd.Close,
                                                  prev_close=dd.prev)).reset_index(drop=True), ['symbol', 'us_date'])
        out.append(f'{name} {last:%m-%d} {len(today)}봉')
    return ', '.join(out)

## 6. 실행 묶음 (서버 일정이 부르는 곳)
`run_step(이름, 날짜)`가 한 단계를 돌리고 `model.runs`에 기록한다. 같은 단계가 겹쳐 돌지 않게 잠그고, 실패하면 텔레그램으로 알린다(`ALERT_TELEGRAM_TOKEN`, `ALERT_TELEGRAM_CHAT_ID`가 있을 때).
`rebuild_from_db(이름)`: 서버 자료 폴더가 없어졌을 때 DB의 원자료로 다시 만든다(미국 시간봉·VIX·코스피200 긴 표는 다시 내려받는다).

In [ ]:
def telegram(text):
    tok, chat = ENV.get('ALERT_TELEGRAM_TOKEN'), ENV.get('ALERT_TELEGRAM_CHAT_ID')
    if not tok or not chat:
        return
    try:
        data = urllib.parse.urlencode(dict(chat_id=chat, text=text[:3500])).encode()
        urllib.request.urlopen(f'https://api.telegram.org/bot{tok}/sendMessage', data=data, timeout=20)
    except Exception:
        pass


def log_run(step, T, started, ok, message):
    try:
        ensure_tables()
        with db() as con:
            con.execute('insert into model.runs (step, t, started_at, finished_at, ok, message) values (%s, %s, %s, now(), %s, %s)',
                        (step, T, started.to_pydatetime(), ok, message[:4000]))
    except Exception as err:
        print('실행 기록 실패', type(err).__name__, flush=True)


STEPS = {'morning0730': lambda T: step_morning(T, '0730'), 'morning0811': lambda T: step_morning(T, '0811'),
         'open': step_open, 'close': step_close,
         'bootstrap': lambda T: str(bootstrap(T)), 'save_boot': lambda T: str(save_folder_db(LIVE / BOOT_NAME)),
         'bootstrap_rest': lambda T: str(bootstrap(T, steps={'premarket', 'market', 'dart'})),
         'night_live': step_night_live, 'us_market': step_us_market}   # 앞 단계가 끝난 뒤 나머지만


def run_step(step, T=None):
    T = T or (last_finished_day() if step.startswith('bootstrap') else today())   # 처음 자료는 끝난 장까지만
    LIVE.mkdir(parents=True, exist_ok=True)
    lock = open(LIVE / f'.lock_{step}', 'w')
    try:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except BlockingIOError:
        print(step, '이미 도는 중이라 건너뜀', flush=True)
        return
    started = now_kst()
    print(f'=== {step} {T} 시작 {started:%H:%M:%S}', flush=True)
    try:
        msg = STEPS[step](T)
        print(f'=== {step} {T} 끝 {now_kst():%H:%M:%S} {msg}', flush=True)
        log_run(step, T, started, True, str(msg))
    except BaseException as err:
        text = f'{type(err).__name__}: {err}\n' + traceback.format_exc()[-1500:]
        print(text, flush=True)
        log_run(step, T, started, False, text)
        telegram(f'[아침 모델] {step} {T} 실패: {type(err).__name__}: {str(err)[:300]}')
        raise SystemExit(1)


def rebuild_from_db(name=BOOT_NAME, until=None):
    """DB 원자료로 자료 폴더를 다시 만든다."""
    out, data = LIVE / name, LIVE / name / 'data'
    data.mkdir(parents=True, exist_ok=True)
    q = lambda sql: pd.read_sql(sql, psycopg.connect(db_url()))
    save_csv(model_universe(), data / 'universe.csv')
    for n in ('fred_release_dates.csv', 'fomc_dates.csv'):
        shutil.copy2(MODEL / 'data' / n, data / n)
    fut = q('select date, session, code, open, high, low, close, volume from model.futures_daily order by date, session')
    save_csv(fut, data / 'futures_daily.csv')
    save_csv(q('select date, code, hhmmss, price, volume from model.night_minutes order by date, hhmmss'), data / 'night_minutes.csv')
    save_csv(q('select date, open as "Open", close as "Close", code from model.stock_daily order by code, date'), data / 'universe_daily.csv')
    save_csv(q('select day, code, hhmmss, price, volume from model.premarket order by day, code, hhmmss'), data / 'universe_premarket.csv')
    save_csv(pd.DataFrame(columns=['date', 'day', 'code', 'hhmmss', 'price', 'volume']), data / 'nxt_premarket.csv')
    k = q('select date, open as "Open", high as "High", low as "Low", close as "Close", volume as "Volume" from model.ks200 order by date')
    save_csv(k.assign(code='KS200', name='코스피200 지수'), data / 'kr_daily.csv')
    _, long = boot_ks200()
    save_csv(long, data / 'ks200_long.csv')
    import collect
    save_csv(collect.collect_us_hourly(), data / 'us_hourly.csv')
    save_csv(boot_vix(), data / 'vix_daily.csv')
    save_csv(investor_placeholder(), data / 'investor_daily.csv')
    save_csv(q('select code, corp_code, rcept_no, rcept_dt, report_nm, corp_cls, flr_nm, rm from model.filings order by code, rcept_dt, rcept_no'), out / 'dart_filings.csv')
    until = until or fut[fut.session == 'day'].date.max()
    log = dict(started_at=now_kst().isoformat(timespec='seconds'), kind='rebuild_from_db', base='DB',
               trading_day=dict(date=until, open=True, checked_by='rebuild_from_db'),
               inputs_sha256={n: lp.sha(data / n) for n in lp.INPUTS}, finished_at=now_kst().isoformat(timespec='seconds'))
    (out / 'refresh_manifest.json').write_text(json.dumps(log, ensure_ascii=False, indent=1))
    return out

## 7. 손으로 돌릴 때 (서버 일정은 이 칸을 건너뛴다)

In [ ]:
# 처음 한 번: 자료 받기 → DB에 넣기
# run_step('bootstrap'); run_step('save_boot')
# 하루 단계를 손으로: run_step('morning0730', '20261121')